# Kernel Functions Profiling

This notebook is there to show how to do some simple function profiling using LISA.

We'll be using the Ftrace function profiler (See "Function profiling" in https://lwn.net/Articles/370423/) for that, and will present the relevant Python APIs from Devlib & LISA that make it easier to use.

In [ ]:
import logging
from lisa.utils import setup_logging
setup_logging()

In [ ]:
import os
from lisa.target import Target, TargetConf

## Target configuration

The only target requirement here is to have enough Ftrace goodies enabled (look at the requirements for **CONFIG_FUNCTION_PROFILER**)

In [ ]:
# target = Target(
#     kind='linux',
#     name='myhikey960',
#     host='192.168.0.1',
#     username='root',
#     password='root',
# )
# Uses settings from target_conf.yml
target = Target.from_default_conf()

## Experiment setup

We can run whatever we want here, let's just build a simple ((1 20% task) x NR_CPUS) workload

In [ ]:
from lisa.wlgen.rta import RTA, RTAPhase, PeriodicWload

In [ ]:
rtapp_profile = {
    f'task{cpu}': RTAPhase(
        prop_wload=PeriodicWload(
            duty_cycle_pct=20,
            period=16e-3,
            duration=1,
        )
    )
    for cpu in range(target.number_of_cpus)
}

In [ ]:
wload = RTA.from_profile(target, rtapp_profile, name='profiling_wload')

Now, let's define the functions we want to do some profiling on. Do keep in mind all functions might not be profilable - that can happen if they are inline.

In [ ]:
functions = [
    "scheduler_tick",
    "run_rebalance_domains"
]

We're using an FtraceCollector so might as well record some basic events to get a meaningful trace

In [ ]:
events = [
    "sched_switch",
    "sched_wakeup",
    "sched_wakeup_new",
    "task_rename",
    
    "funcgraph_entry",
    "funcgraph_exit",
]

## Running the experiment

In [ ]:
from lisa.trace import Trace, FtraceCollector

In [ ]:
trace_path = os.path.join(wload.res_dir, 'trace.dat')
# In order to be able to collect the profiling information, the kernel must have been compiled with:
# CONFIG_FUNCTION_TRACER=y
# CONFIG_FUNCTION_PROFILER=y
ftrace_coll = FtraceCollector(target, events=events, functions=functions, buffer_size=10240, output_path=trace_path)

with wload, ftrace_coll:
    wload.run()


In [ ]:
!tree {wload.res_dir}

## Using function grapher events


The kernel function grapher emits ``funcgraph_entry`` and ``funcgraph_exit`` events that can be used by LISA to compute profiling statistics and compare them.

In [ ]:
import holoviews as hv
hv.extension('bokeh')

trace = Trace(
    trace_path,
    plat_info=target.plat_info,
    df_fmt='polars-lazyframe',
)

In [ ]:
# Get a lisa.stats.Stats instance loaded with all the profiling information
stat_funcs = {'mean': None, 'count': None, 'std': None}
# metric is either "self_time" for the time spent in the function or "cum_time" for the cumulative time
# spent in the function and in all its children as well
stats = trace.ana.functions.profile_stats(metric='self_time', functions=functions, stats=stat_funcs)
stats.df

In [ ]:
# Use filename="..." to save to a file
fig = stats.plot_stats()
fig

### Compare profiling data from two traces

In [ ]:
# "Create" two traces out of the first one by splitting it in half.
# In real life, it's likely you would have 2 different traces
half = trace.start + trace.time_range / 2
trace_ref = trace[:half]
trace_other = trace[half:]


stat_funcs = {'mean': None, 'count': None}

stats = trace_ref.ana.functions.compare_with_traces(
    [trace_other],
    metric='self_time',
    functions=functions,
    stats=stat_funcs,
)
stats.df

In [ ]:
# Stats from both traces side by side
fig = stats.plot_stats(compare=False)
fig

In [ ]:
# Compare the information from the 2 traces
fig = stats.plot_stats(compare=True)
fig